In [ ]:
# 0. 작업 준비: 필요한 라이브러리 임포트
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Keras 관련 임포트 통일
from keras.models import Model
from keras.layers import (
    Input, Dense, Dropout, GlobalAveragePooling2D,
    Conv2D, BatchNormalization, MaxPooling2D, Flatten,
    Activation
)
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.losses import SparseCategoricalCrossentropy
from keras.datasets import cifar10
from keras.layers import Resizing, RandomFlip, RandomRotation, RandomZoom, RandomContrast

In [ ]:
# 1. 데이터 로드 및 분할
(x_data, y_data), (tt_x, tt_y) = cifar10.load_data()

tr_x, val_x, tr_y, val_y = train_test_split(
    x_data, y_data,
    train_size=0.8,
    stratify=y_data,
    random_state=42
)

In [ ]:
# 2. 데이터 증강 및 전처리 파이프라인 구성 (tf.data 활용)
# 💡 참고: 직접 만드는 CNN은 EfficientNet만큼 크지 않으므로, 이미지 크기를 줄이면(예: 96)
# 훈련 속도가 빨라지고 과적합 방지에 도움이 될 수 있습니다. 여기서는 160을 유지합니다.
IMG_SIZE = 32
BATCH_SIZE = 256
AUTOTUNE = tf.data.AUTOTUNE

# 이미지 크기 조정 레이어
resizing_layer = Resizing(IMG_SIZE, IMG_SIZE)

# 데이터 증강 레이어
data_augmentation = tf.keras.Sequential([
    RandomFlip("horizontal"),
    RandomRotation(factor=0.1),
    RandomZoom(height_factor=0.1),
    RandomContrast(factor=0.1)
], name="data_augmentation")

# 데이터셋 전처리 함수
def prepare_dataset(x, y, training=False):
    # 직접 만든 CNN이므로 EfficientNet 전처리 대신 간단한 정규화를 사용합니다.
    x = tf.cast(x, tf.float32) / 255.0
    x = resizing_layer(x) # 크기 조정
    if training:
        x = data_augmentation(x) # 훈련 시에만 증강 적용
    return x, y

# tf.data 파이프라인 구축
train_ds = tf.data.Dataset.from_tensor_slices((tr_x, tr_y))
train_ds = train_ds.map(lambda x, y: prepare_dataset(x, y, training=True), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.cache().shuffle(1000).batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_x, val_y))
val_ds = val_ds.map(prepare_dataset, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.cache().batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((tt_x, tt_y))
test_ds = test_ds.map(prepare_dataset, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.cache().batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)

print("✅ CNN 모델을 위한 tf.data 파이프라인이 구성되었습니다.")

In [ ]:
from keras.regularizers import l2

def build_custom_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    """사용자 정의 CNN 모델을 빌드하는 함수"""
    input_tensor = Input(shape=input_shape)

    # --- 특징 추출기 (Feature Extractor) ---
    # Block 1: Conv x2, Filters=64
    x = Conv2D(64, (3, 3), padding='same', kernel_regularizer=l2(1e-4))(input_tensor)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(64, (3, 3), padding='same', kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.3)(x)

    # Block 2: Conv x2, Filters=128
    x = Conv2D(128, (3, 3), padding='same', kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(128, (3, 3), padding='same', kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.4)(x)

    # Block 3: Conv x2, Filters=256
    x = Conv2D(256, (3, 3), padding='same', kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(256, (3, 3), padding='same', kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.5)(x)

    # --- 분류기 (Classifier) ---
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    output_tensor = Dense(10, activation='softmax')(x)

    model = Model(inputs=input_tensor, outputs=output_tensor)
    return model

# 모델 생성 및 요약 출력
model = build_custom_cnn()
model.summary()

In [ ]:
# 4. 모델 컴파일 및 훈련 (학습률 스케줄러 추가)
import math

# --- 하이퍼파라미터 설정 ---
EPOCHS = 100
LEARNING_RATE = 1e-3
WARMUP_EPOCHS = 5 # 처음 5 에포크 동안 학습률을 점진적으로 증가

# --- 코사인 감쇠를 사용한 학습률 스케줄러 정의 ---
def cosine_decay_with_warmup(epoch):
    """Warmup 후 Cosine Decay를 적용하는 학습률 스케줄러 함수"""
    if epoch < WARMUP_EPOCHS:
        # 선형적으로 학습률 증가 (Warmup)
        return (epoch + 1) / WARMUP_EPOCHS * LEARNING_RATE
    else:
        # 코사인 함수 형태로 학습률 감소 (Cosine Decay)
        effective_epoch = epoch - WARMUP_EPOCHS
        total_decay_epochs = EPOCHS - WARMUP_EPOCHS
        cosine_decay = 0.5 * (1 + math.cos(math.pi * effective_epoch / total_decay_epochs))
        return LEARNING_RATE * cosine_decay

# LearningRateScheduler 콜백 생성
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(cosine_decay_with_warmup, verbose=1)

# --- 모델 컴파일 ---
loss_fn = SparseCategoricalCrossentropy()
optimizer = Adam(learning_rate=LEARNING_RATE) # 초기 학습률 설정

model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# --- 콜백 설정 ---
# ReduceLROnPlateau는 이제 새로운 스케줄러로 대체되었으므로 제거합니다.
es = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1) # patience를 조금 늘려줍니다.
ck = ModelCheckpoint('best_custom_cnn_scheduled.keras', monitor='val_accuracy', save_best_only=True, verbose=1)

print("\n--- CNN 모델 학습 시작 (학습률 스케줄러 적용) ---")
history = model.fit(train_ds,
                    validation_data=val_ds,
                    epochs=EPOCHS,
                    callbacks=[es, ck, lr_scheduler])

# 학습 결과 시각화
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
# 5. 최종 모델 평가
print("\n--- 최종 성능 평가 ---")
model.load_weights('best_custom_cnn_scheduled.keras')
loss, acc = model.evaluate(test_ds)
print(f'최종 테스트 데이터 손실: {loss:.4f}')
print(f'최종 테스트 데이터 정확도: {acc:.4f}')

In [ ]:
# 📦 라이브러리 임포트
import math
import tensorflow as tf
from keras.datasets import cifar10
from keras.models import Sequential
from keras.layers import (Conv2D, BatchNormalization, Activation,
                           MaxPooling2D, Dropout, GlobalAveragePooling2D,
                           Dense, Input)
from keras.optimizers import Adam
from keras.losses import SparseCategoricalCrossentropy
from keras.callbacks import EarlyStopping, ModelCheckpoint, LearningRateScheduler
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.regularizers import l2
from sklearn.model_selection import train_test_split

# 📥 데이터 로딩 및 분할
(x, y), (_, _) = cifar10.load_data()
x_data, tt_x, y_data, tt_y = train_test_split(x, y, train_size=0.85, stratify=y)
tr_x, val_x, tr_y, val_y = train_test_split(x_data, y_data, train_size=0.82, stratify=y_data)
 
# 🎨 정규화
s_tr_x = tr_x / 255.0 
s_val_x = val_x / 255.0
s_tt_x = tt_x / 255.0

# 🧠 모델 구조 정의
def conv_block(model, filters, dropout_rate):
    model.add(Conv2D(filters, 3, padding='same', kernel_regularizer=l2(1e-4)))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Conv2D(filters, 3, padding='same', kernel_regularizer=l2(1e-4)))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(2))
    model.add(Dropout(dropout_rate))
    return model

model = Sequential()
model.add(Input(shape=(32, 32, 3)))
model = conv_block(model, 64, 0.3)
model = conv_block(model, 128, 0.4)
model = conv_block(model, 256, 0.5)
model.add(GlobalAveragePooling2D())
model.add(Dense(128, activation='relu', kernel_regularizer=l2(1e-4)))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))

# ⚙️ 컴파일
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 64

   # 데이터 증강 레이어 정의
data_augmentation = tf.keras.Sequential([
   tf.keras.layers.RandomFlip("horizontal"),
   tf.keras.layers.RandomRotation(0.1),
   tf.keras.layers.RandomZoom(0.1),
])
   
   # 데이터셋 전처리 함수
def prepare(x, y, augment=False):
    x = tf.cast(x, tf.float32) / 255.0
    if augment:
        x = data_augmentation(x)
    return x, y

def cosine_decay_with_warmup(epoch, lr_max=0.001, lr_min=1e-6, warmup_epochs=5, total_epochs=100):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs * lr_max
    else:
        effective_epoch = epoch - warmup_epochs
        decay_epochs = total_epochs - warmup_epochs
        cosine_decay = 0.5 * (1 + math.cos(math.pi * effective_epoch / decay_epochs))
        return (lr_max - lr_min) * cosine_decay + lr_min
    
   # tf.data.Dataset 객체 생성
train_ds = tf.data.Dataset.from_tensor_slices((tr_x, tr_y))
train_ds = train_ds.cache().shuffle(1000).map(lambda x, y: prepare(x, y, True), num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)
 
val_ds = tf.data.Dataset.from_tensor_slices((val_x, val_y))
val_ds = val_ds.map(prepare, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).cache().prefetch(buffer_size=AUTOTUNE)
    
test_ds = tf.data.Dataset.from_tensor_slices((tt_x, tt_y))
test_ds = test_ds.map(prepare, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).cache().prefetch(buffer_size=AUTOTUNE)

# 📍 콜백 설정
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_accuracy', verbose=1),
    LearningRateScheduler(cosine_decay_with_warmup, verbose=1)
]

# 🚀 학습
history = model.fit(train_ds, 
                    validation_data=val_ds,
                    epochs=100,
                    callbacks=callbacks)

# 🧾 테스트 평가
test_loss, test_acc = model.evaluate(s_tt_x, tt_y)
print(f"✅ 테스트 정확도:  {test_acc:.4f}, 손실: {test_loss:.4f}")